# Exploration of WVV ÖPNV data using the GTFS database

No real goal, just exploring the data while printing, plotting and saving as much relevant information as possible.

In [3]:
import pandas as pd

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## GTFS Static - Extraction

### Extracting all routes managed by WVV

In [4]:
df_agency = pd.read_csv("../data_static/agency.txt")
df_wvv = df_agency.loc[df_agency["agency_name"] == "Würzburger Verkehrsverbund"]
df_wvv

,agency_id,agency_name,agency_url,agency_timezone,agency_lang
290,360,Würzburger Verkehrsverbund,https://www.bahn.de,Europe/Berlin,de


In [5]:
df_routes = pd.read_csv("../data_static/routes.txt")

wvv_id = df_wvv.loc[290, "agency_id"]
df_routes_wvv = df_routes.loc[df_routes["agency_id"] == wvv_id]
df_routes_wvv = df_routes_wvv.drop("agency_id", axis=1).reset_index(drop=True)

print(f"Total number of routes managed by WVV: {df_routes_wvv.shape[0]}")

df_routes_wvv.head()


Total number of routes managed by WVV: 90


,route_long_name,route_short_name,route_type,route_id,route_color,route_text_color
0,NaN,1,0,8263,NaN,NaN
1,NaN,10,3,10285,NaN,NaN
2,NaN,11,3,21987,NaN,NaN
3,NaN,114,3,17677,NaN,NaN
4,NaN,12,3,8479,NaN,NaN


In [6]:
print(f"Unique long route names: {df_routes_wvv["route_long_name"].unique()}")
print(f"Unique route colors: {df_routes_wvv["route_color"].unique()}")
print(f"Unique route text colors: {df_routes_wvv["route_text_color"].unique()}")

Unique long route names: [nan]
Unique route colors: <StringArray>
[nan]
Length: 1, dtype: str
Unique route text colors: <StringArray>
[nan]
Length: 1, dtype: str


As shown above `route_long_name`, `route_color` and `route_text_color` contains no information, hence will be dropped.

In [7]:
df_routes_wvv = df_routes_wvv.drop(["route_long_name", "route_color", "route_text_color"], axis=1)

df_routes_wvv.to_csv("../data_extract/routes_wvv.csv", index=False)

df_routes_wvv.head()

,route_short_name,route_type,route_id
0,1,0,8263
1,10,3,10285
2,11,3,21987
3,114,3,17677
4,12,3,8479


### Differentiating route types

In [8]:
print(f"Unique route types: {df_routes_wvv["route_type"].unique()}")

Unique route types: [0 3]


In [9]:
df_routes_wvv_0 = df_routes_wvv.loc[df_routes_wvv["route_type"] == 0]
df_routes_wvv_0

,route_short_name,route_type,route_id
0,1,0,8263
21,3,0,14250
25,4,0,18905
50,5,0,15397
56,53,0,17923
57,54,0,14669


In [10]:
df_routes_wvv_3 = df_routes_wvv.loc[df_routes_wvv["route_type"] == 3]
df_routes_wvv_3.head()

,route_short_name,route_type,route_id
1,10,3,10285
2,11,3,21987
3,114,3,17677
4,12,3,8479
5,13,3,6662


While it is not specified to what the `route_type` columns refers to, I highly assume that `route_type = 0` are Straßenbahnen, while `route_type = 3` are buses.

Wierdly, the Straßenbahnen `53` and `54` are not regular Straßenbahnen and the Straßebahn with number `2` is missing.

### Extracting all the trips managed by WVV

In [11]:
df_trips = pd.read_csv("../data_static/trips.txt")
df_trips_wvv = df_trips.loc[df_trips["route_id"].isin(df_routes_wvv["route_id"])]

print(f"Total number of trips managed by WVV: {df_trips_wvv.shape[0]}")

df_trips_wvv.head()

Total number of trips managed by WVV: 6808


,route_id,service_id,trip_id
21400,10285,129,1027601
21401,10285,129,1029179
21402,10285,129,112921
21403,10285,129,1182997
21404,10285,129,1261651


In [12]:
df_trips_wvv = df_trips_wvv.merge(df_routes_wvv, on="route_id")

df_trips_wvv.to_csv("../data_extract/trips_wvv.csv", index=False)

df_trips_wvv.head()

,route_id,service_id,trip_id,route_short_name,route_type
0,10285,129,1027601,10,3
1,10285,129,1029179,10,3
2,10285,129,112921,10,3
3,10285,129,1182997,10,3
4,10285,129,1261651,10,3


### Extracting all the stop times managed by WVV

In [13]:
# Reading this file takes a few seconds, due to its size: hence seperate cell
df_stoptimes = pd.read_csv("../data_static/stop_times.txt")

In [14]:
df_stoptimes_wvv = df_stoptimes.merge(df_trips_wvv, on="trip_id", how="inner")

df_stoptimes_wvv.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,route_id,service_id,route_short_name,route_type
0,1535467,12:57:00,12:57:00,247879,0,NaN,NaN,14250,1517,3,0
1,1535467,12:58:00,13:00:00,536012,1,NaN,NaN,14250,1517,3,0
2,1535467,13:01:00,13:02:00,211932,2,NaN,NaN,14250,1517,3,0
3,1535467,13:03:00,13:03:00,591397,3,NaN,NaN,14250,1517,3,0
4,1535467,13:04:00,13:05:00,364013,4,NaN,NaN,14250,1517,3,0


In [15]:
print(f"Unique pickup types: {df_stoptimes_wvv["pickup_type"].unique()}")
print(f"Unique dropoff types: {df_stoptimes_wvv["drop_off_type"].unique()}")
print(f"Unique service IDs: {df_stoptimes_wvv["service_id"].unique()}")

Unique pickup types: [nan  1.]
Unique dropoff types: [nan]
Unique service IDs: [1517  138 1192  993 1602  129  524  304  438 1515  786 1719 1531 1377
 1655  885 1938  250 1416 1277 1390 1346 1143 1716 1410 1142  644 1794
  420 1627  955  828  767 1391 1673  337 2017]


As shown above `drop_off_type` contains no information, hence will be dropped.

In [16]:
df_stoptimes_wvv = df_stoptimes_wvv.drop("drop_off_type", axis=1)

df_stoptimes_wvv.to_csv("../data_extract/stoptimes_wvv.csv", index=False)

df_stoptimes_wvv.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,route_id,service_id,route_short_name,route_type
0,1535467,12:57:00,12:57:00,247879,0,NaN,14250,1517,3,0
1,1535467,12:58:00,13:00:00,536012,1,NaN,14250,1517,3,0
2,1535467,13:01:00,13:02:00,211932,2,NaN,14250,1517,3,0
3,1535467,13:03:00,13:03:00,591397,3,NaN,14250,1517,3,0
4,1535467,13:04:00,13:05:00,364013,4,NaN,14250,1517,3,0


### Extracting all the stops managed by WVV

In [17]:
df_stops = pd.read_csv("../data_static/stops.txt")

stops_wvv_ids = df_stoptimes_wvv["stop_id"].unique()
df_stops_wvv = df_stops.loc[df_stops["stop_id"].isin(stops_wvv_ids)].reset_index(drop=True)

print(f"Number of stops managed by WVV: {df_stops_wvv.shape[0]}")

df_stops_wvv.head()

Number of stops managed by WVV: 1614


,stop_name,parent_station,stop_id,stop_lat,stop_lon,location_type,platform_code
0,Acholshausen Acholshausen,610766.0,307344,49.645252,9.998132,NaN,NaN
1,Acholshausen Acholshausen,610766.0,93234,49.645176,9.997971,NaN,NaN
2,Albertsh. Fuchsstadter Str.,266584.0,632185,49.693928,9.932169,NaN,12
3,Albertsh. Lindflurer Str.,529523.0,177682,49.694230,9.927417,NaN,12
4,Albertsh. Lindflurer Str.,529523.0,445232,49.694195,9.927336,NaN,11


In [18]:
print(f"Unique location types: {df_stops_wvv["location_type"].unique()}")

Unique location types: [nan]


As shown above `location_type` contains no information, hence will be dropped.

In [19]:
df_stops_wvv = df_stops_wvv.drop("location_type", axis=1)

df_stops_wvv.to_csv("../data_extract/stops_wvv.csv", index=False)

df_stops_wvv.head()

,stop_name,parent_station,stop_id,stop_lat,stop_lon,platform_code
0,Acholshausen Acholshausen,610766.0,307344,49.645252,9.998132,NaN
1,Acholshausen Acholshausen,610766.0,93234,49.645176,9.997971,NaN
2,Albertsh. Fuchsstadter Str.,266584.0,632185,49.693928,9.932169,12
3,Albertsh. Lindflurer Str.,529523.0,177682,49.694230,9.927417,12
4,Albertsh. Lindflurer Str.,529523.0,445232,49.694195,9.927336,11


## GTFS Static - Plotting

In [1]:
from folium_plots import plot_stops_map

### Vizualizing all Stops

In [4]:
df_stops_wvv = pd.read_csv("../data_extract/stops_wvv.csv")

plot_stops_map(df_stops_wvv, f"stops_map")

Map saved as stops_map.html. Open this file in your web browser!


Since there are so many stops, this is quite laggy. Here we will group all stops that have the same name (eg. Hauptbahnhof ZOB has all platforms as different stops)

In [5]:
df_stops_wvv["mean_lat"] = df_stops_wvv.groupby("stop_name")["stop_lat"].transform("mean")
df_stops_wvv["mean_lon"] = df_stops_wvv.groupby("stop_name")["stop_lon"].transform("mean")

df_stops_wvv_grouped = df_stops_wvv.drop_duplicates(subset="stop_name").reset_index(drop=True)
df_stops_wvv_grouped = df_stops_wvv_grouped[["stop_name", "stop_id", "mean_lat", "mean_lon"]].rename({"mean_lat": "stop_lat", "mean_lon": "stop_lon"}, axis=1)

plot_stops_map(df_stops_wvv_grouped, f"stops_map_grouped")

Map saved as stops_map_grouped.html. Open this file in your web browser!


### Vizualizing how often Stops are visited

In [6]:
df_stoptimes_wvv = pd.read_csv("../data_extract/stoptimes_wvv.csv", dtype={"route_short_name": str})

df_stopvisits = df_stoptimes_wvv.groupby("stop_id").size().reset_index(name="count")
df_stopvisits.head()

df_stopvisits = df_stopvisits.merge(df_stops_wvv_grouped, on="stop_id", how="inner")
df_stopvisits = df_stopvisits[["stop_name", "stop_id", "count", "stop_lat", "stop_lon"]]

min_val = df_stopvisits["count"].min()
max_val = df_stopvisits["count"].max()
        
y_vals = 255 - (df_stopvisits["count"] - min_val) / (max_val - min_val) * 255
df_stopvisits["color"] = y_vals.apply(lambda y: f"rgb(255, {int(y)}, 0)")

plot_stops_map(df_stopvisits, f"stops_map_grouped_heat", heat=True)

Map saved as stops_map_grouped_heat.html. Open this file in your web browser!
